In [25]:
import sys
import os

# ensure we're always running from project root
os.chdir(os.path.dirname(os.getcwd()))
sys.path.insert(0, '.')

In [26]:
from src.load import load_data
from src.clean import clean_data
import pandas as pd

df_raw = load_data()
df = clean_data(df_raw)
df = df[df['Customer ID'].notna()][~df['Invoice'].str.startswith('C')]

── Data loaded ──────────────────────────
Rows:              1,033,019
Date range:        2009-12-01 → 2011-12-09
Unique customers:  5,940
Cancellations:     19,100
Null Customer IDs: 235,150
─────────────────────────────────────────
── Cleaning summary ─────────────────────
Rows before: 1,033,019
Rows after:  1,021,374
Removed:     11,645
Outlier rows flagged: 7
─────────────────────────────────────────


C:\Users\TomTurner\AppData\Local\Temp\ipykernel_22816\1847685586.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[df['Customer ID'].notna()][~df['Invoice'].str.startswith('C')]


## Customer's first purchase month

In [27]:
cohort_df = df.groupby(by='Customer ID', as_index=False)['InvoiceDate'].min()
cohort_df['Cohort'] = cohort_df['InvoiceDate'].dt.month
cohort_df.drop(columns=['InvoiceDate'], inplace=True)

cohort_df.head(10)

,Customer ID,Cohort
0,12346,3
1,12347,10
2,12348,9
3,12349,4
4,12350,2
5,12351,11
6,12352,11
7,12353,10
8,12354,4
9,12355,5


## Assign cohort

In [28]:
df = pd.merge(df, cohort_df, on='Customer ID', how='left')
df.sample(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice,IsOutlier,Cohort
576828,561222,21703,BAG 125g SWIRLY MARBLES,12,2011-07-26 09:27:00,0.42,13588,United Kingdom,5.04,False,3
709323,574920,23569,TRADTIONAL ALPHABET STAMP SET,8,2011-11-07 16:34:00,4.95,13985,United Kingdom,39.60,False,2
472640,547799,20726,LUNCH BAG WOODLAND,30,2011-03-25 12:48:00,1.65,17139,United Kingdom,49.50,False,1
369714,534555,22487,WHITE WOOD GARDEN PLANT LADDER,4,2010-11-23 12:22:00,8.50,17508,Greece,34.00,False,12
240147,520398,22189,CREAM HEART CARD HOLDER,3,2010-08-25 16:11:00,3.95,13713,United Kingdom,11.85,False,5
406963,539215,22816,CARD MOTORBIKE SANTA,36,2010-12-16 12:42:00,0.42,15574,United Kingdom,15.12,False,9
15575,491123,22042,CHRISTMAS CARD SINGING ANGEL,36,2009-12-09 15:20:00,0.42,17043,United Kingdom,15.12,False,12
664960,570672,21715,GIRLS VINTAGE TIN SEASIDE BUCKET,8,2011-10-11 14:52:00,2.55,12536,France,20.40,False,10
593206,563205,22667,RECIPE BOX RETROSPOT,2,2011-08-14 12:03:00,2.95,14040,United Kingdom,5.90,False,12
311806,528335,22204,MILK PAN BLUE POLKADOT,4,2010-10-21 13:45:00,3.75,12400,Australia,15.00,False,10


## Calculating the period number